# Assignment 3: Implement a Novel Attention Variant from Description (100 points)

In this assignment, you will practice the paper-to-implementation skill by implementing a novel attention mechanism from a research-style description.

## Background: Neighborhood Attention

Standard self-attention allows every token to attend to every other token, giving $O(L^2)$ complexity. **Neighborhood Attention (NA)** restricts each token to attend only to its $k$ nearest neighbors in the sequence, reducing complexity to $O(Lk)$.

However, unlike local window attention, NA uses a **learned offset** to dynamically shift the attention window for each token.

### Method Description

Given input $X \in \mathbb{R}^{B \times L \times d}$ and window size $w$ (odd integer):

1. **Projections:** $Q = XW_Q$, $K = XW_K$, $V = XW_V$ where $W_Q, W_K \in \mathbb{R}^{d \times d_k}$, $W_V \in \mathbb{R}^{d \times d_v}$

2. **Neighborhood extraction:** For each position $i$, extract the local window of keys and values:
   $$K_i^{\text{local}} = K[i - w//2 : i + w//2 + 1] \in \mathbb{R}^{w \times d_k}$$
   $$V_i^{\text{local}} = V[i - w//2 : i + w//2 + 1] \in \mathbb{R}^{w \times d_v}$$
   Use zero-padding for positions near the boundaries.

3. **Local attention:** For each position $i$:
   $$A_i = \text{softmax}\left(\frac{Q_i (K_i^{\text{local}})^T}{\sqrt{d_k}}\right) \in \mathbb{R}^{1 \times w}$$
   $$O_i = A_i V_i^{\text{local}} \in \mathbb{R}^{1 \times d_v}$$

4. **Output projection:** $Y = OW_O$ where $W_O \in \mathbb{R}^{d_v \times d}$

**Implementation note:** The neighborhood extraction can be implemented efficiently using `F.unfold` or manual padding + slicing. For this assignment, a loop-based implementation is acceptable, but a vectorized implementation using `F.pad` and `unfold` is preferred.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

---

> **WARNING:** Do not modify any code outside of the designated solution areas.

---

## Part 1: Shape Analysis (8 points)

**[Non-coding]** For $B = 2$, $L = 16$, $d = 64$, $d_k = d_v = 32$, $w = 5$:

1. (2 points) What are the shapes of $Q$, $K$, $V$ after projection?
2. (2 points) What is the shape of $K_i^{\text{local}}$ for a single position $i$?
3. (2 points) What is the shape of the attention scores $A_i$ for a single position?
4. (2 points) What is the total number of attention score computations across all positions? Compare to standard attention.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 2: Neighborhood Extraction (15 points)

**[Coding]** Implement a function that extracts local neighborhoods from a sequence.

Given a tensor of shape $(B, L, d)$ and window size $w$, return a tensor of shape $(B, L, w, d)$ where `output[b, i, :, :]` contains the $w$ neighbors of position $i$ (zero-padded at boundaries).

**Hint:** Use `F.pad` to add $w//2$ zeros on each side, then use `.unfold()` along the sequence dimension.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def extract_neighborhoods(x, window_size):
    """
    Extract local neighborhoods from a sequence.
    
    Args:
        x: (B, L, d)
        window_size: odd integer w
    
    Returns:
        neighborhoods: (B, L, w, d)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 3: Single-Head Neighborhood Attention (15 points)

**[Coding]** Implement `NeighborhoodAttention` as an `nn.Module`.

Use your `extract_neighborhoods` function from Part 2.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class NeighborhoodAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, window_size=5):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        # x: (B, L, d)
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 4: Smoke Test (5 points)

**[Coding]** Verify your implementation works with a shape test.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Smoke test: B=2, L=16, d=64, d_k=d_v=32, w=5
# Verify output shape is (2, 16, 64)

""" END OF THIS PART """

## Part 5: Multi-Head Extension (12 points)

**[Coding]** Extend your implementation to multi-head neighborhood attention.

**Specification:**
- $h$ heads, each with dimension $d_k / h$ and $d_v / h$
- All heads use the same window size $w$
- Concatenate head outputs and project with $W_O$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MultiHeadNeighborhoodAttention(nn.Module):
    def __init__(self, d_model, num_heads=4, window_size=5):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 6: Complexity Analysis (8 points)

**[Non-coding]** Analyze the computational complexity.

1. (3 points) What is the time complexity of standard multi-head attention in terms of $B$, $L$, $d$, and $h$?
2. (3 points) What is the time complexity of multi-head neighborhood attention in terms of $B$, $L$, $d$, $h$, and $w$?
3. (2 points) For $L = 4096$ and $w = 15$, how much faster is neighborhood attention compared to standard attention (ratio of FLOPs)?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 7: Build a Transformer Block (10 points)

**[Coding]** Build a complete pre-norm transformer block using `MultiHeadNeighborhoodAttention`.

- Pre-norm residual connections
- LayerNorm
- FFN: Linear(d, 4d) → GELU → Linear(4d, d)
- Dropout with probability $p = 0.1$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class NATransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads=4, window_size=5, dropout=0.1):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 8: Sequence Classification Model (10 points)

**[Coding]** Build a complete sequence classification model using NA transformer blocks.

**Architecture:**
- Token embedding: `nn.Embedding(vocab_size, d_model)`
- Positional embedding: learnable `nn.Embedding(max_len, d_model)`
- $N$ NA transformer blocks
- Mean pooling over sequence length
- Classification head: Linear(d_model, num_classes)

Use `vocab_size=1000`, `d_model=64`, `num_heads=4`, `window_size=7`, `num_layers=3`, `num_classes=5`, `max_len=128`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class NAClassifier(nn.Module):
    def __init__(self, vocab_size=1000, d_model=64, num_heads=4,
                 window_size=7, num_layers=3, num_classes=5, max_len=128):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        # x: (B, L) integer token IDs
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 9: Training on Synthetic Data (10 points)

**[Coding]** Train the model on a synthetic classification task.

Generate synthetic data:
- 1000 sequences of length 32, with random token IDs from [0, 999]
- Labels: class is determined by the majority token value modulo 5
- Train/test split: 800/200
- Train for 20 epochs with Adam, lr=1e-3, batch_size=32
- Report training loss and test accuracy per epoch

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Generate data, train, evaluate

""" END OF THIS PART """

## Part 10: Limitations and Extensions (7 points)

**[Non-coding]**

1. (3 points) What is the main limitation of fixed-window neighborhood attention compared to standard attention? Give a concrete example of a task where NA would perform poorly.

2. (2 points) Propose a modification to NA that could address this limitation while maintaining sub-quadratic complexity.

3. (2 points) How would you adapt neighborhood attention for 2D data (e.g., images represented as a flattened sequence of patches)?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """